# Day 3: Repeating until something happens, and when the time step is too big

**Time:** about 1 hour  |  **Needs:** Days 1 and 2
**Reservoir idea:** a model with a limit (food, or reservoir energy) and a time step that can be too large for it
**Python you will learn:** updating variables, `while` loops, `break` and `continue`, loop patterns (counting, summing, finding a maximum), looping with an index

By the end of today you will be able to:

1. update a variable in place (`total += rate`) and use it as a counter or an accumulator,
2. choose between a `for` loop (known number of repeats) and a `while` loop (repeat until a condition),
3. stop a loop early with `break` and skip a pass with `continue`,
4. write the standard loops that count, sum and find a maximum or minimum,
5. explain why a time step that is too large makes a numerical solution oscillate, and test for it.

> **Companion reading (optional, about 40 minutes).** Charles Severance, *Python for Everybody* (free online: <https://www.py4e.com/html3/>), Chapter 5 "Iterations": "Updating variables", "The while statement", "Infinite loops" (it includes break), "Finishing iterations with continue", "Definite loops using for", "Loop patterns", "Debugging".

## 0. Setup and recap

Run the next cell. It loads the tools and re-creates the Day 2 functions that we need (they are given here, so you do not have to copy them from yesterday's notebook).

**Recap of Day 2.** `if`/`elif`/`else` choose between actions; a function takes parameters and `return`s a result; `for dt in [..]` loops over the items of a list.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

alpha = 1.74      # growth constant, per hour
N0 = 100          # starting number of bacteria


def rel_error_percent(approx, exact):
    """Relative error of approx compared with exact, in percent."""
    return abs(approx - exact) / abs(exact) * 100


def euler_growth(alpha, N0, dt, t_end):
    """Numerical solution of dN/dt = alpha*N with time step dt. Returns (times, values)."""
    n_steps = round(t_end / dt)
    times = [0.0]
    values = [N0]
    for n in range(1, n_steps + 1):
        values.append((1 + alpha * dt) * values[-1])
        times.append(n * dt)
    return times, values

print("ready")

## 1. Story: how long until the economic limit?

*(An invented but typical situation.)* A well produces 1000 STB/day and its rate falls by about 8 % each month. The company stops a well when it makes less than 100 STB/day, because below that it costs more to run than it earns. The manager asks a simple question: **"In how many months will the well reach its economic limit?"**

You could work it out with logarithms, but a reservoir simulator cannot: it does not know in advance how many steps the answer needs. It just keeps stepping *until* something happens. A loop that repeats until a condition is met is called a `while` loop. Today's lesson uses it, and then looks at what happens when the steps themselves are too big.

## 2. Updating variables

In maths, `x = x + 1` is impossible. In Python it is perfectly sensible: *take the current value of `x`, add 1, and store the result back in `x`*. The variable must already exist.

```python
count = 0
count = count + 1        # now 1
count += 1               # shortcut for count = count + 1, now 2
total *= 2               # shortcut for total = total * 2
```

Two roles you will see constantly:

- a **counter** starts at 0 and goes up by 1 each time something happens,
- an **accumulator** (or running total) starts at 0 and adds a value each time.

Predict what is printed, then run the cell.

In [ ]:
n = 5
n = n + 2
n *= 3
n -= 1
print(n)

**Your turn 2.1.** A well produced `q1 = 2450`, `q2 = 2410` and `q3 = 2395` STB on three consecutive days. Start with `cum = 0` and add each day's production to it using `+=` (one line per day). After the three lines `cum` must hold the cumulative production.

In [ ]:
q1 = 2450
q2 = 2410
q3 = 2395

cum = 0
# YOUR CODE HERE

In [ ]:
# Check (do not edit)
assert cum == 7255, f"cum should be 7255 (2450 + 2410 + 2395), got {cum}"
print("Correct")

<details><summary>Solution</summary>

```python
cum += q1
cum += q2
cum += q3
```
</details>

## 3. The `while` loop

```python
while condition:
    # these indented lines repeat as long as the condition is True
```

Python checks the condition **before every pass**. If it is `False` the loop stops (even before the first pass). Something inside the loop must eventually make the condition `False`, otherwise the loop never ends.

Here is the manager's question. The rate starts at 1000 and is multiplied by 0.92 each month (an 8 % decline). We count months until the rate drops below the limit of 100. Predict the answer first (is it about 10 months? 28? 100?), then run.

In [ ]:
q = 1000.0           # STB/day, current rate
limit = 100.0        # economic limit
months = 0           # counter

while q > limit:
    q = q * 0.92     # rate after one more month of decline
    months += 1      # one more month has passed

print(f"The rate falls below {limit:.0f} STB/day after {months} months (rate {q:.1f})")

The loop ran 28 times, and nobody had to say "28" in advance. Compare with the `for` loop:

| | `for` | `while` |
|---|---|---|
| use when | you know how many times (or have a list to go through) | you repeat *until a condition* is met |
| example | 40 time steps | until the rate falls below the limit |

### Infinite loops

If nothing in the loop changes the condition, it runs forever. For example, deleting the line `q = q * 0.92` leaves `q` at 1000 for ever. If a cell seems stuck (the play button keeps spinning), press the **stop** button (in JupyterLab: *Kernel → Interrupt Kernel*). To be safe, loops that might not finish often carry a counter and give up after a maximum number of passes.

**Your turn 3.1.** A different well starts at `q = 2000` STB/day and declines by 10 % per month (multiply by 0.90). The limit is 150 STB/day. Use a `while` loop to find the number of months until the rate falls below the limit. Store it in `months`.

In [ ]:
q = 2000.0
limit = 150.0
months = 0

# YOUR CODE HERE

In [ ]:
# Check (do not edit)
assert months == 25, f"months should be 25, got {months}. Check the condition (q > limit) and that you update both q and months in the loop"
assert q < 150, "q should be below the limit when the loop ends"
print("Correct:", months, "months")

<details><summary>Solution</summary>

```python
while q > limit:
    q = q * 0.90
    months += 1
```
</details>

## 4. `break` and `continue`

- `break` **leaves the loop immediately**.
- `continue` **skips the rest of this pass** and goes on to the next one.

Both work in `for` and in `while` loops. A very common use of `break` is *"look through the data and stop at the first one that matches"*:

```python
for rate in rates:
    if rate < limit:
        print("below the limit:", rate)
        break
```

Often you also need to know *where* the match is. For that, loop over the **positions** with `range(len(my_list))`, then read the item as `my_list[i]`:

```python
for i in range(len(rates)):
    print(i, rates[i])
```

Run it to see how `range(len(...))` and the index fit together:

In [ ]:
rates = [900, 850, 700]
for i in range(len(rates)):
    print("position", i, "holds", rates[i])

**Your turn 4.1.** The list `pressures` holds the average reservoir pressure (psi) at the end of each month. Find the **first month (position in the list)** in which the pressure is below `p_min = 2500`. Store the position in `first_below` and stop looking as soon as you find it. If the pressure never falls below, `first_below` must stay `-1`.

In [ ]:
pressures = [3000, 2940, 2880, 2810, 2740, 2650, 2560, 2470, 2380, 2300]
p_min = 2500

first_below = -1
# YOUR CODE HERE

In [ ]:
# Check (do not edit)
assert first_below == 7, f"first_below should be 7 (the value 2470 is at position 7), got {first_below}"
print("Correct: position", first_below, "holds", pressures[first_below])

<details><summary>Solution</summary>

```python
for i in range(len(pressures)):
    if pressures[i] < p_min:
        first_below = i
        break
```
</details>

`continue` is useful for skipping unwanted items. This loop adds up only the producing days of a list where `0` means "well shut in":

In [ ]:
daily = [2450, 0, 2395, 0, 2360]
total = 0
for rate in daily:
    if rate == 0:
        continue          # skip the shut-in days
    total += rate
print(total)

## 5. Loop patterns: counting, summing, finding the biggest

Most loops over data are one of a few patterns. Each one sets up a variable *before* the loop and updates it *inside*.

| Pattern | Before the loop | Inside the loop |
|---|---|---|
| count | `count = 0` | `count += 1` (often inside an `if`) |
| sum | `total = 0` | `total += value` |
| maximum | `best = first item` | `if value > best: best = value` |
| minimum | `best = first item` | `if value < best: best = value` |

We will practise on a record of daily oil rates, where `0` means the well was shut in that day.

In [ ]:
daily_rates = [2450, 2410, 2395, 0, 2360, 2330, 2310, 0, 2285, 2260]
print(len(daily_rates), "days in the record")

**Your turn 5.1.** Using one `for` loop, find for the **producing days only** (rate above 0):

- `n_days`: how many producing days there were,
- `total`: the cumulative production,
- `average`: the average rate on producing days (`total / n_days`, calculated *after* the loop).

In [ ]:
n_days = 0
total = 0

# YOUR CODE HERE
average = None

In [ ]:
# Check (do not edit)
assert n_days == 8, f"n_days should be 8 (two days have rate 0), got {n_days}"
assert total == 18800, f"total should be 18800, got {total}"
assert average == 2350, f"average should be 2350, got {average}"
print("Correct")

<details><summary>Solution</summary>

```python
for rate in daily_rates:
    if rate > 0:
        n_days += 1
        total += rate
average = total / n_days
```
</details>

**Your turn 5.2.** The list `water_cuts` holds the monthly water cut of a well. Without using `max()`, find the highest value (`highest`) and the position where it occurs (`highest_day`). Start with the first item as the best so far.

In [ ]:
water_cuts = [0.12, 0.15, 0.31, 0.28, 0.47, 0.44, 0.63, 0.61]

highest = water_cuts[0]
highest_day = 0
# YOUR CODE HERE

In [ ]:
# Check (do not edit)
assert highest == 0.63, f"highest should be 0.63, got {highest}"
assert highest_day == 6, f"highest_day should be 6, got {highest_day}"
assert highest == max(water_cuts), "this should agree with the built-in max()"
print("Correct. The built-ins max(), min(), sum() and len() do these patterns for you once you understand them:")
print(max(water_cuts), min(water_cuts), sum(daily_rates), len(daily_rates))

<details><summary>Solution</summary>

```python
for i in range(len(water_cuts)):
    if water_cuts[i] > highest:
        highest = water_cuts[i]
        highest_day = i
```
</details>

**Your turn 5.3.** Write a function `cumulative(values)` that returns a **new list of running totals**: the first item is the first value, the second is the sum of the first two, and so on. For `[1, 2, 3]` it returns `[1, 3, 6]`. (This is how a simulator turns a rate history into cumulative production, `Np`.)

In [ ]:
def cumulative(values):
    """List of running totals of values."""
    # YOUR CODE HERE
    pass

In [ ]:
# Check (do not edit)
assert cumulative([1, 2, 3]) == [1, 3, 6], f"cumulative([1, 2, 3]) should be [1, 3, 6], got {cumulative([1, 2, 3])}"
assert cumulative([]) == [], "an empty list should give an empty list"
assert cumulative(daily_rates)[-1] == 18800, "the last running total must equal the sum of all values"
print("Correct")

<details><summary>Solution</summary>

```python
def cumulative(values):
    """List of running totals of values."""
    result = []
    total = 0
    for v in values:
        total += v
        result.append(total)
    return result
```
</details>

## 6. Time stepping with `while`: a trap

On Day 1 the loop was `for n in range(1, n_steps + 1)` with `time = n * dt`. Why not the more obvious `while t < t_end` with `t = t + dt`? Predict how many steps the next loop takes when we step from 0 to 1 in steps of 0.1, then run it.

In [ ]:
t = 0.0
dt = 0.1
steps = 0
while t < 1.0:
    t = t + dt
    steps += 1
print("steps:", steps, " final t:", t)

It took **11** steps, not 10. After ten additions, `t` is `0.9999999999999999`, not exactly 1.0 (Day 2's decimal surprise again), so `t < 1.0` is still true and the loop does one more pass. The cure is to stop a tiny margin early:

```python
while t < t_end - 1e-9:
```

**Your turn 6.1.** Write `count_steps(t_end, dt)`: a function with a `while` loop that steps `t` from 0 to `t_end` in steps of `dt` and returns the number of steps. Use the margin above.

In [ ]:
def count_steps(t_end, dt):
    """Number of time steps of size dt needed to reach t_end."""
    # YOUR CODE HERE
    pass

In [ ]:
# Check (do not edit)
assert count_steps(1.0, 0.1) == 10, f"count_steps(1.0, 0.1) should be 10, got {count_steps(1.0, 0.1)}"
assert count_steps(2.0, 0.05) == 40, "count_steps(2.0, 0.05) should be 40"
assert count_steps(5.0, 0.01) == 500, "count_steps(5.0, 0.01) should be 500"
print("Correct")

<details><summary>Solution</summary>

```python
def count_steps(t_end, dt):
    """Number of time steps of size dt needed to reach t_end."""
    t = 0.0
    steps = 0
    while t < t_end - 1e-9:
        t = t + dt
        steps += 1
    return steps
```
</details>

(The simpler way, `round(t_end / dt)`, is what `euler_growth` uses. Both are fine; now you know why.)

## 7. A model with a limit: the logistic equation

Back to the bacteria. The model $dN/dt = \alpha N$ lets the colony grow forever. A real colony runs out of food. Add a limit $K$, the largest colony the food can support:

$$\frac{dN}{dt} = \alpha N\left(1-\frac{N}{K}\right) \tag{5}$$

When $N$ is small, $1 - N/K$ is about 1 and growth is as before. As $N$ approaches $K$, the bracket goes to 0 and growth stops. The numerical recipe changes in **one line**:

$$N^{n+1} = N^{n} + \Delta t\;\alpha N^{n}\left(1-\frac{N^{n}}{K}\right) \tag{6}$$

(Compare with Day 1: before it was $N^{n+1}=N^n+\Delta t\,\alpha N^n$.)

**Your turn 7.1.** Write `euler_logistic(alpha, N0, K, dt, t_end)`, similar to `euler_growth`, using equation (6). It returns the lists `times` and `values`.

In [ ]:
def euler_logistic(alpha, N0, K, dt, t_end):
    """Numerical solution of the logistic equation (5). Returns (times, values)."""
    n_steps = round(t_end / dt)
    times = [0.0]
    values = [N0]
    # YOUR CODE HERE
    return times, values

In [ ]:
# Check (do not edit)
ts, ns = euler_logistic(1.74, 100, 1000, 0.01, 5.0)
assert len(ts) == 501, f"Expected 501 points, got {len(ts)}"
assert abs(ns[1] - (100 + 0.01 * 1.74 * 100 * (1 - 100 / 1000))) < 1e-9, "the first step is wrong: use N + dt * alpha * N * (1 - N / K)"
assert 990 < ns[-1] < 1001, f"final value {ns[-1]:.1f}; the colony should level off close to K = 1000"
print(f"Correct: the colony levels off near K = 1000 (got {ns[-1]:.1f})")

<details><summary>Solution</summary>

```python
for n in range(1, n_steps + 1):
    N_old = values[-1]
    N_new = N_old + dt * alpha * N_old * (1 - N_old / K)
    values.append(N_new)
    times.append(n * dt)
```
</details>

In [ ]:
K = 1000
ts, ns = euler_logistic(alpha, N0, K, 0.01, 5.0)

plt.plot(ts, ns)
plt.axhline(K, color="gray", linestyle="--", label="K (food limit)")
plt.xlabel("time (h)")
plt.ylabel("number of bacteria")
plt.legend()
plt.grid(True)
plt.show()

An S-shaped curve: fast growth at first, then it levels off at $K$. This is the shape of many things in reservoir engineering: pressure support running out, saturation approaching its end point, recovery factor against time.

## 8. When the time step is too large

### Predict first

Same model, same $K$, but now a **huge** time step, $\Delta t = 1.3$ h. What will the curve do? (a) the same S-curve, a bit rougher; (b) overshoot $K$ once and then settle; (c) something else. Write your guess below, then run the next cell.

In [ ]:
# my guess:

In [ ]:
ts, ns = euler_logistic(alpha, N0, K, 1.3, 15.0)
print([round(v) for v in ns[:12]])

plt.plot(ts, ns, "o-")
plt.axhline(K, color="gray", linestyle="--", label="K")
plt.xlabel("time (h)")
plt.ylabel("number of bacteria")
plt.legend()
plt.grid(True)
plt.show()

The numbers jump above and below $K$ and **never settle**. The real colony does not do this: the oscillation is created by the numerical method, not by the physics. This is **numerical instability**.

### Why it happens

Suppose $N$ is slightly off the limit: $N^n = K(1+\varepsilon)$, where $\varepsilon$ is a small relative error. Put it into equation (6). The bracket $1-N/K$ becomes $-\varepsilon$, so

$$N^{n+1} \approx K(1+\varepsilon) + \Delta t\,\alpha K(1+\varepsilon)(-\varepsilon) \approx K\Big(1 + \big(1-\alpha\Delta t\big)\,\varepsilon\Big)$$

(we dropped $\varepsilon^2$ because it is tiny). In words: **each step multiplies the distance from $K$ by the factor $1-\alpha\Delta t$.**

- If $|1-\alpha\Delta t| < 1$, the distance shrinks every step: the solution settles. (Stable.)
- If $|1-\alpha\Delta t| > 1$, the distance grows every step: the solution blows away from $K$. (Unstable.)

With $\alpha = 1.74$ and $\Delta t = 1.3$: $1 - 2.262 = -1.262$, with size 1.26 > 1, so unstable. The stability limit is $\alpha\Delta t < 2$, i.e. $\Delta t < 2/\alpha \approx 1.15$ h. When the factor is negative (here $\alpha\Delta t > 1$) the solution jumps from one side of $K$ to the other each step, which is the zig-zag you saw.

**Your turn 8.1.** Write `is_stable(alpha, dt)`, which returns `True` if the factor above has size less than 1 and `False` otherwise. (Use `abs`.)

In [ ]:
def is_stable(alpha, dt):
    """True if the explicit logistic step is stable for this alpha and dt."""
    # YOUR CODE HERE
    pass

In [ ]:
# Check (do not edit)
assert is_stable(1.74, 0.5) is True, "dt = 0.5 should be stable (factor 1 - 0.87 = 0.13)"
assert is_stable(1.74, 1.1) is True, "dt = 1.1 should be stable (factor -0.914, size below 1)"
assert is_stable(1.74, 1.3) is False, "dt = 1.3 should be unstable (factor -1.262)"
assert is_stable(1.74, 2.0) is False, "dt = 2.0 should be unstable"
print("Correct")

<details><summary>Solution</summary>

```python
def is_stable(alpha, dt):
    """True if the explicit logistic step is stable for this alpha and dt."""
    factor = 1 - alpha * dt
    return abs(factor) < 1
```
</details>

Now a use for a `while` loop. Imagine a simulation that starts with a large time step, and whenever it is unstable, **halves** the step and tries again. How many halvings are needed, and what step do we end with?

**Your turn 8.2.** Start with `dt_safe = 4.0` and `halvings = 0`. While `is_stable(alpha, dt_safe)` is `False`, halve `dt_safe` and add 1 to `halvings`. (Use `not` in the condition.)

In [ ]:
dt_safe = 4.0
halvings = 0

# YOUR CODE HERE

In [ ]:
# Check (do not edit)
assert halvings == 2, f"halvings should be 2 (4.0 -> 2.0 -> 1.0), got {halvings}"
assert dt_safe == 1.0, f"dt_safe should be 1.0, got {dt_safe}"
print(f"Correct: dt = {dt_safe} h after {halvings} halvings")

<details><summary>Solution</summary>

```python
while not is_stable(alpha, dt_safe):
    dt_safe = dt_safe / 2
    halvings += 1
```
</details>

### Stable is not the same as accurate

Does `dt = 1.0` give a good answer? Compare it with a very fine run (the reference) and with a small step:

In [ ]:
ref_t, ref_n = euler_logistic(alpha, N0, K, 0.001, 6.0)      # very small step = reference
plt.plot(ref_t, ref_n, "k", linewidth=2, label="reference (dt = 0.001)")

for dt in [0.1, 1.0]:
    t_, n_ = euler_logistic(alpha, N0, K, dt, 6.0)
    plt.plot(t_, n_, "o-", markersize=4, label=f"dt = {dt}   stable: {is_stable(alpha, dt)}")

plt.axhline(K, color="gray", linestyle="--")
plt.xlabel("time (h)")
plt.ylabel("number of bacteria")
plt.legend()
plt.grid(True)
plt.show()

At $\Delta t = 1.0$ the solution is stable: it ends at $K$ and stays there. But compare it with the reference. After 1 hour it has about 257 bacteria where the reference has about 388, so it lags badly at first. By hour 3 it has overshot $K$ slightly (about 1010 against 954 for the reference). The steps are too coarse to follow the shape of the curve. **Stability stops the solution from exploding; accuracy needs a smaller step still.** Choosing $\Delta t$ means looking at both.

## 9. Bridge: a `while` loop is the heart of `kusim`

Open `src/kusim/simulator.py`. The method that runs a whole simulation is this:

```python
def run(self):
    while not self.finished:
        self.step()
    return self.results()
```

It is the loop from Section 3: *keep taking steps until the simulation is finished.* The property `finished` is `True` when the end time is reached **or a stop condition was met**. One stop condition in the code is the well's *minimum oil rate*: when the oil rate falls to the economic limit, the run stops, just like the well in our story. The `step()` method does one IMPES time step, the numerical recipe of Day 8.

The instability of this lesson is also in the real simulator. The explicit saturation update (the equation we looked at on Day 1) can oscillate when the time step is too large for the flow rates and the block sizes. The factor $1-\alpha\Delta t$ has a counterpart there, built from the flow through a block, the time step and the block's pore volume. The thesis example at $\Delta t = 1$ day oscillates for that reason (`docs/porting-notes.md` records it).

## 10. Self-quiz

Write short answers as comments, then open the answers.

1. `count = 0`, then `count += 1` runs three times. What is `count`?
2. When do you choose `while` instead of `for`?
3. What does `break` do? What does `continue` do?
4. A list `[4, 9, 2]`. For the maximum pattern, what do you set `best` to before the loop, and why not 0?
5. For the logistic model, what does $|1-\alpha\Delta t|>1$ mean for the solution?

In [ ]:
# 1.
# 2.
# 3.
# 4.
# 5.

<details><summary>Answers</summary>

1. 3.
2. When you do not know in advance how many repeats are needed, and want to repeat *until* a condition is met (e.g. a rate falls below a limit).
3. `break` leaves the loop completely. `continue` skips the rest of the current pass and goes on to the next one.
4. The first item (`4`). Starting at 0 would give a wrong answer if all values are negative (e.g. a pressure drop), and it ties the code to the data. The first item is always a valid starting point.
5. The distance from $K$ grows at every step: the numerical solution is unstable and oscillates away from the true answer.
</details>

## 11. Pocket reference (today's Python)

| You want to... | Write |
|---|---|
| update a variable | `x = x + 1`  or  `x += 1`, `x *= 2` |
| repeat until a condition fails | `while condition:` (indented block) |
| stop a loop now / skip this pass | `break` / `continue` |
| loop with the position | `for i in range(len(my_list)):` then `my_list[i]` |
| count | `count = 0` ... `count += 1` |
| sum | `total = 0` ... `total += value` |
| maximum | `best = my_list[0]` ... `if v > best: best = v` |
| built-in shortcuts | `sum(x)`, `max(x)`, `min(x)`, `len(x)` |
| loop that stops if the answer is not reached | add a pass counter and `break` after a maximum |
| make a condition false | `not condition` |

## 12. Wrap-up

Today you:

- updated variables in place and used them as counters and accumulators,
- wrote `while` loops for "repeat until", and found where `for` is better,
- used `break` and `continue`, and looped over the positions of a list,
- wrote the count, sum, maximum and running-total loop patterns,
- met a model with a limit (the logistic equation) and saw its numerical solution **oscillate** when $\Delta t$ was too big,
- derived a stability rule, wrote `is_stable`, and learned that stable does not mean accurate,
- saw that the main loop of the simulator is a `while` loop with a stop condition.

**Next, Day 4:** strings and reservoir basics: porosity, pore volume, original oil in place, and the oilfield-unit conversions. You will parse text such as `PERMX 150 md` and write functions with several arguments.